# MVP — Machine Learning & Analytics

**Nome:** Yasmin Cristina Gonçalves dos Santos  
**Data:** 05/07/2026  
**Dataset:** Processos judiciais brasileiros sobre violência de gênero/feminicídio, com dados do Portal CNJ e da API Pública do DataJud/CNJ  
**Tipo de problema:** Classificação binária

## Observações importantes

Este notebook foi feito para o MVP da disciplina de Machine Learning & Analytics. A ideia foi seguir um fluxo completo de Machine Learning: entender o problema, carregar os dados, fazer uma análise inicial, preparar a base, treinar modelos, avaliar os resultados e comentar as limitações.

Tentei manter o projeto simples e reprodutível, usando bibliotecas conhecidas e sem depender de upload manual no Colab.

# 1. Definição do problema

## 1.1 Descrição do problema

Neste trabalho, eu uso dados de processos judiciais relacionados a feminicídio. O problema escolhido foi tentar prever se um processo em primeiro grau terá uma decisão judicial substantiva em até 730 dias depois do ajuizamento.

Na prática, isso significa transformar o problema em uma classificação com duas respostas possíveis: sim ou não.

Esse tipo de análise pode interessar a pessoas que estudam dados jurídicos, políticas públicas ou o funcionamento do sistema de justiça. Mesmo assim, é importante deixar claro que o modelo não deve ser usado para tomar decisões sobre processos individuais.

## 1.2 Objetivo do MVP

O objetivo deste MVP é criar e comparar modelos de Machine Learning para prever se um processo relacionado a feminicídio terá decisão substantiva em até 730 dias, usando informações disponíveis no início do processo.

## 1.3 Tipo de problema

**Tipo escolhido:** classificação binária.

**Justificativa:** o resultado que quero prever tem apenas duas classes:

- `1`: teve decisão substantiva em até 730 dias;
- `0`: não teve decisão substantiva nesse prazo.

Por isso, o problema é de classificação.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**

- Algumas informações iniciais do processo, como tribunal, classe, sistema, formato e assuntos, podem ajudar o modelo.
- Como as classes não estão perfeitamente balanceadas, a acurácia não deve ser a única métrica analisada.
- Processos muito recentes, sem tempo suficiente de acompanhamento, não devem entrar na modelagem principal.

**Critérios de sucesso:**

- Métrica principal: F1-score da classe positiva.
- O modelo precisa superar o baseline (`DummyClassifier`).
- O notebook precisa rodar no Colab de forma simples.

# 2. Ambiente, bibliotecas e reprodutibilidade

Nesta parte, importo as bibliotecas usadas no notebook e defino a seed para facilitar a reprodução dos resultados.

In [1]:
# === Setup básico e reprodutibilidade ===
import io
import math
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")

URL_PUBLICA_BASE = "https://raw.githubusercontent.com/yas-cgs/mvp-ml-analytics-feminicidio/main/data/mvp/base_modelagem.csv.gz"

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Seed global:", SEED)

Python: 3.11.14
pandas: 2.3.3
Seed global: 42


## 2.1 Dependências adicionais

Não usei bibliotecas muito específicas. A ideia foi manter o projeto com ferramentas comuns, como `pandas`, `numpy`, `matplotlib`, `seaborn`, `requests` e `scikit-learn`. Para rodar localmente, as dependências também estão no arquivo `requirements.txt`.

In [ ]:
# Nenhuma instalação extra é necessária no Colab padrão.
# Para execução local, use: pip install -r requirements.txt

# 3. Seleção e carga dos dados

## 3.1 Fonte dos dados

Os dados usados neste projeto vêm de duas fontes principais:

- `portal_cnj.csv`: arquivo exportado do Portal CNJ, com uma linha por caso. Ele tem informações como tribunal, município, unidade judicial, número do processo, descrição da decisão e status de confidencialidade. Foram considerados apenas casos não confidenciais.
- API Pública do DataJud/CNJ: API do CNJ usada no projeto original por meio do script `datajud_requester.py`. Ela traz metadados dos processos e o histórico de movimentações.

Na pasta deste MVP, os dados originais já estão em formato Parquet:

- `Data/processos.parquet`;
- `Data/movimentos.parquet`.

Para o notebook rodar no Colab, eu criei uma base menor, anonimizada e compactada chamada `data/mvp/base_modelagem.csv.gz`.

**Observação importante:** a API Pública do DataJud não disponibiliza processos sigilosos. Como processos de feminicídio e violência doméstica podem tramitar sob sigilo, esta base representa apenas os processos públicos disponíveis, e não todos os casos existentes.

Também é importante reforçar que este modelo não prevê culpa, inocência ou mérito jurídico. Ele serve apenas para uma análise acadêmica e agregada.

## 3.2 Carga dos dados

A base é carregada por uma URL pública do GitHub. Assim, o notebook não depende de upload manual ou de arquivos no meu computador.

In [3]:
# === Carga da base pública ===
resposta = requests.get(URL_PUBLICA_BASE, timeout=60)
resposta.raise_for_status()

df = pd.read_csv(io.BytesIO(resposta.content), compression="gzip")
df.head()

HTTPError: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/yas-cgs/mvp-ml-analytics-feminicidio/main/data/mvp/base_modelagem.csv.gz

In [ ]:
# === Verificações iniciais ===
print("Formato da base:", df.shape)
print("\nTipos das variáveis:")
display(df.dtypes.rename("tipo").to_frame())
print("\nValores ausentes por coluna:")
display(df.isna().sum().rename("ausentes").to_frame())

variaveis_dicionario = pd.DataFrame([
    ("tribunal", "Tribunal de origem do registro público."),
    ("classe_codigo", "Código da classe processual no DataJud."),
    ("sistema_nome", "Sistema processual informado no início do registro."),
    ("formato_nome", "Formato do processo, como eletrônico ou físico."),
    ("ano_ajuizamento", "Ano do ajuizamento."),
    ("mes_ajuizamento", "Mês do ajuizamento."),
    ("orgao_julgador_municipio_ibge", "Código IBGE do município do órgão julgador, quando disponível."),
    ("quantidade_assuntos", "Quantidade de assuntos associados ao processo."),
    ("assunto_homicidio_qualificado", "Indicador de assunto Homicídio Qualificado."),
    ("assunto_violencia_domestica_mulher", "Indicador de assuntos de violência doméstica/contra a mulher."),
    ("assunto_crime_tentado", "Indicador de assunto Crime Tentado."),
    ("assunto_ameaca", "Indicador de assunto Ameaça."),
    ("assunto_homicidio_simples", "Indicador de assunto Homicídio Simples."),
    ("assunto_arma", "Indicador textual agregado de assuntos relacionados a armas."),
    ("assunto_medida_protetiva", "Indicador textual agregado de medida protetiva."),
    ("decisao_substantiva_ate_730d", "Alvo: decisão substantiva em até 730 dias."),
], columns=["variável", "descrição"])
variaveis_dicionario

A tabela acima confirma que a base pública já está reduzida aos campos necessários para a modelagem. Identificadores como número do processo, `id`, nomes de órgão julgador, datas de decisão e campos de atualização não foram incluídos.

# 4. Análise exploratória dos dados

Nesta etapa, faço uma análise inicial da base. O objetivo é entender o tamanho dos dados, a distribuição da variável-alvo e algumas características importantes antes de treinar os modelos.

In [ ]:
display(df.head())
print("Dimensão:", df.shape)
print("Duplicidades completas:", df.duplicated().sum())

resumo_numerico = df.describe(include="all").T
resumo_numerico

Essa primeira visualização ajuda a conferir se a base foi carregada corretamente e se as variáveis estão no formato esperado.

In [ ]:
target = "decisao_substantiva_ate_730d"
contagem_alvo = df[target].value_counts().sort_index()
percentual_alvo = (df[target].value_counts(normalize=True).sort_index() * 100).round(2)
distribuicao_alvo = pd.DataFrame({"processos": contagem_alvo, "percentual": percentual_alvo})
distribuicao_alvo.index = ["0 - sem decisão em até 730 dias", "1 - com decisão em até 730 dias"]
display(distribuicao_alvo)

ax = sns.countplot(data=df, x=target, hue=target, palette="Set2", legend=False)
ax.set_title("Distribuição da variável-alvo")
ax.set_xlabel("Decisão substantiva em até 730 dias")
ax.set_ylabel("Quantidade de processos")
plt.show()

A variável-alvo está desbalanceada, pois a classe `0` aparece mais vezes que a classe `1`. Por isso, além da acurácia, também serão usadas métricas como F1, recall e balanced accuracy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ordem_tribunais = df["tribunal"].value_counts().head(12).index
sns.countplot(data=df[df["tribunal"].isin(ordem_tribunais)], y="tribunal", order=ordem_tribunais, ax=axes[0], color="#4C78A8")
axes[0].set_title("Processos por tribunal - 12 mais frequentes")
axes[0].set_xlabel("Quantidade de processos")
axes[0].set_ylabel("Tribunal")

por_ano = df.groupby("ano_ajuizamento")[target].agg(processos="size", taxa_positiva="mean").reset_index()
sns.barplot(data=por_ano, x="ano_ajuizamento", y="processos", ax=axes[1], color="#72B7B2")
axes[1].set_title("Processos por ano de ajuizamento")
axes[1].set_xlabel("Ano de ajuizamento")
axes[1].set_ylabel("Quantidade de processos")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

por_ano.tail(15)

Os gráficos mostram que alguns tribunais têm mais registros na base e que a distribuição muda ao longo dos anos. Isso é importante para interpretar os resultados do modelo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(data=df, x="quantidade_assuntos", bins=20, ax=axes[0, 0], color="#F58518")
axes[0, 0].set_title("Distribuição da quantidade de assuntos")
axes[0, 0].set_xlabel("Quantidade de assuntos")
axes[0, 0].set_ylabel("Processos")

classe_top = df["classe_codigo"].astype("string").value_counts().head(10)
classe_top.plot(kind="bar", ax=axes[0, 1], color="#54A24B")
axes[0, 1].set_title("Classes processuais mais frequentes")
axes[0, 1].set_xlabel("Código da classe")
axes[0, 1].set_ylabel("Processos")
axes[0, 1].tick_params(axis="x", rotation=45)

sistema_top = df["sistema_nome"].fillna("Ausente").value_counts().head(10)
sistema_top.plot(kind="bar", ax=axes[1, 0], color="#B279A2")
axes[1, 0].set_title("Sistemas processuais mais frequentes")
axes[1, 0].set_xlabel("Sistema")
axes[1, 0].set_ylabel("Processos")
axes[1, 0].tick_params(axis="x", rotation=45)

assuntos_flags = [
    "assunto_homicidio_qualificado",
    "assunto_violencia_domestica_mulher",
    "assunto_crime_tentado",
    "assunto_ameaca",
    "assunto_homicidio_simples",
    "assunto_arma",
    "assunto_medida_protetiva",
]
df[assuntos_flags].mean().sort_values().plot(kind="barh", ax=axes[1, 1], color="#E45756")
axes[1, 1].set_title("Proporção de indicadores de assuntos")
axes[1, 1].set_xlabel("Proporção")
axes[1, 1].set_ylabel("Indicador")
plt.tight_layout()
plt.show()

Os indicadores de assunto ajudam a transformar a lista de assuntos em variáveis simples para o modelo. O município do órgão julgador tem muitas categorias, então as categorias raras são agrupadas no pré-processamento.

## 4.1 Síntese da análise exploratória

A base tem desbalanceamento entre as classes, porque a classe `0` aparece mais vezes que a classe `1`. Também existem tribunais com mais registros do que outros e variáveis categóricas com muitas categorias, como o município do órgão julgador.

Por isso, na modelagem eu preciso ter cuidado com categorias raras e usar métricas além da acurácia.

# 5. Preparação dos dados e divisão treino/teste

Aqui eu defino quais colunas entram no modelo, qual é a variável-alvo e como os dados serão separados em treino e teste.

In [ ]:
PROBLEM_TYPE = "classificacao"
features = [col for col in df.columns if col != target]

X = df[features].copy()
y = df[target].astype(int).copy()

ordenado = df.sort_values(["ano_ajuizamento", "mes_ajuizamento"]).reset_index(drop=True)
corte = int(len(ordenado) * 0.80)
treino = ordenado.iloc[:corte].copy()
teste = ordenado.iloc[corte:].copy()

X_train = treino[features].copy()
y_train = treino[target].astype(int).copy()
X_test = teste[features].copy()
y_test = teste[target].astype(int).copy()

print("Treino:", X_train.shape, "| Teste:", X_test.shape)
print("Distribuição treino:")
display(y_train.value_counts(normalize=True).rename("proporcao").to_frame())
print("Distribuição teste:")
display(y_test.value_counts(normalize=True).rename("proporcao").to_frame())

split_anos = pd.DataFrame({
    "conjunto": ["treino", "teste"],
    "ano_min": [treino["ano_ajuizamento"].min(), teste["ano_ajuizamento"].min()],
    "ano_max": [treino["ano_ajuizamento"].max(), teste["ano_ajuizamento"].max()],
    "processos": [len(treino), len(teste)],
    "taxa_positiva": [y_train.mean(), y_test.mean()],
})
split_anos

A divisão temporal deixa o teste mais parecido com uma situação real, pois o modelo treina com dados mais antigos e é avaliado em dados mais recentes.

## 5.1 Justificativa da divisão

Eu usei uma divisão temporal: os processos mais antigos ficaram no treino e os mais recentes ficaram no teste.

Escolhi essa estratégia porque, em uma situação real, o modelo seria usado para processos futuros. Então faz sentido testar o modelo em dados mais recentes do que os usados no treinamento.

# 6. Pré-processamento e pipeline

Nesta etapa, preparo os dados para os modelos. Usei `Pipeline` e `ColumnTransformer` porque isso ajuda a evitar vazamento de dados e deixa o processo mais organizado.

As principais etapas foram:

- preencher valores ausentes;
- padronizar variáveis numéricas;
- transformar variáveis categóricas com One-Hot Encoding;
- agrupar categorias raras.

In [ ]:
numeric_features = [
    "ano_ajuizamento",
    "mes_ajuizamento",
    "quantidade_assuntos",
    "assunto_homicidio_qualificado",
    "assunto_violencia_domestica_mulher",
    "assunto_crime_tentado",
    "assunto_ameaca",
    "assunto_homicidio_simples",
    "assunto_arma",
    "assunto_medida_protetiva",
]

categorical_features = [
    "tribunal",
    "classe_codigo",
    "sistema_nome",
    "formato_nome",
    "orgao_julgador_municipio_ibge",
]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30, sparse_output=False)),
])

preprocess = ColumnTransformer(transformers=[
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

print("Variáveis numéricas:", numeric_features)
print("Variáveis categóricas:", categorical_features)

## 6.1 Decisões de pré-processamento

A preparação foi feita de forma simples. Usei variáveis que já estavam disponíveis no início do processo, como ano e mês do ajuizamento, tribunal, classe, sistema, formato, quantidade de assuntos e alguns indicadores de assunto.

Não usei informações que aconteceriam no futuro do processo, como data da decisão, tempo até a decisão ou movimentos posteriores. Também não usei número do processo, nomes de pessoas ou identificadores.

# 7. Baseline e modelos candidatos

Antes de testar modelos mais elaborados, usei um baseline com `DummyClassifier`. Ele serve como referência mínima, porque basicamente prevê a classe mais frequente.

Depois, comparei esse baseline com alguns modelos de classificação.

## 7.1 Justificativa dos modelos

Escolhi modelos simples e disponíveis no `scikit-learn`:

- Regressão Logística;
- Random Forest;
- Extra Trees;
- HistGradientBoosting.

A ideia foi comparar modelos diferentes, mas sem usar técnicas mais complexas como Deep Learning, porque o problema é tabular e não exige esse tipo de abordagem.

# 8. Treinamento e avaliação inicial

Nesta etapa, treino o baseline e os modelos candidatos usando a mesma divisão de treino e teste. Assim, a comparação fica mais justa.

In [ ]:
def metricas_classificacao(modelo, X_avaliacao, y_avaliacao, prefixo):
    y_pred = modelo.predict(X_avaliacao)
    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_avaliacao)[:, 1]
        roc_auc = roc_auc_score(y_avaliacao, proba)
    else:
        roc_auc = np.nan
    return {
        "conjunto": prefixo,
        "acuracia": accuracy_score(y_avaliacao, y_pred),
        "precisao": precision_score(y_avaliacao, y_pred, zero_division=0),
        "recall": recall_score(y_avaliacao, y_pred, zero_division=0),
        "f1": f1_score(y_avaliacao, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_avaliacao, y_pred),
        "roc_auc": roc_auc,
    }

modelos = {
    "DummyClassifier": Pipeline([
        ("pre", preprocess),
        ("model", DummyClassifier(strategy="most_frequent", random_state=SEED)),
    ]),
    "Regressão Logística": Pipeline([
        ("pre", preprocess),
        ("model", LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")),
    ]),
    "Random Forest": Pipeline([
        ("pre", preprocess),
        ("model", RandomForestClassifier(n_estimators=160, max_depth=14, min_samples_leaf=3, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample")),
    ]),
    "Extra Trees": Pipeline([
        ("pre", preprocess),
        ("model", ExtraTreesClassifier(n_estimators=180, max_depth=16, min_samples_leaf=3, random_state=SEED, n_jobs=-1, class_weight="balanced")),
    ]),
    "HistGradientBoosting": Pipeline([
        ("pre", preprocess),
        ("model", HistGradientBoostingClassifier(max_iter=140, learning_rate=0.06, random_state=SEED)),
    ]),
}

resultados = []
modelos_treinados = {}
for nome, modelo in modelos.items():
    inicio = time.time()
    modelo.fit(X_train, y_train)
    tempo = time.time() - inicio
    modelos_treinados[nome] = modelo
    for conjunto, X_av, y_av in [("treino", X_train, y_train), ("teste", X_test, y_test)]:
        linha = metricas_classificacao(modelo, X_av, y_av, conjunto)
        linha["modelo"] = nome
        linha["tempo_treino_s"] = round(tempo, 3)
        resultados.append(linha)

resultados_modelos = pd.DataFrame(resultados)
comparacao_teste = resultados_modelos.query("conjunto == 'teste'").sort_values("f1", ascending=False)
display(comparacao_teste)
resultados_modelos

A tabela compara os modelos no conjunto de teste. A métrica principal será o F1-score da classe positiva, porque a classe `1` é menos frequente e é importante equilibrar precisão e recall.

## 8.1 Análise dos resultados iniciais

A tabela mostra o desempenho dos modelos no conjunto de teste. Como as classes estão desbalanceadas, olho principalmente para o F1-score, mas também considero recall, precisão, balanced accuracy e ROC-AUC.

# 9. Validação e otimização de hiperparâmetros

Nesta etapa, faço um ajuste simples de hiperparâmetros usando `GridSearchCV`. A busca é feita apenas nos dados de treino, para não usar o teste na escolha do modelo.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
modelo_tuning = Pipeline([
    ("pre", preprocess),
    ("model", LogisticRegression(max_iter=1200, random_state=SEED)),
])

grade = {
    "model__C": [0.1, 1.0, 3.0],
    "model__class_weight": [None, "balanced"],
}

busca = GridSearchCV(
    estimator=modelo_tuning,
    param_grid=grade,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)

inicio = time.time()
busca.fit(X_train, y_train)
tempo_tuning = time.time() - inicio

print("Hiperparâmetros testados:", grade)
print("Métrica de seleção: f1")
print("Melhor configuração:", busca.best_params_)
print("Melhor F1 médio em validação cruzada:", round(busca.best_score_, 4))
print("Tempo de busca (s):", round(tempo_tuning, 3))

melhor_lr = busca.best_estimator_
linha_treino = metricas_classificacao(melhor_lr, X_train, y_train, "treino")
linha_teste = metricas_classificacao(melhor_lr, X_test, y_test, "teste")
for linha in (linha_treino, linha_teste):
    linha["modelo"] = "Regressão Logística Otimizada"
    linha["tempo_treino_s"] = round(tempo_tuning, 3)

resultados_tuning = pd.DataFrame([linha_treino, linha_teste])
display(resultados_tuning)

resultados_modelos = pd.concat([resultados_modelos, resultados_tuning], ignore_index=True)
comparacao_final = resultados_modelos.query("conjunto == 'teste'").sort_values("f1", ascending=False).reset_index(drop=True)
comparacao_final

O conjunto de teste foi usado apenas no final, depois da escolha dos hiperparâmetros.

## 9.1 Discussão da otimização

A busca foi pequena para o notebook continuar rápido. Mesmo assim, ela ajuda a testar algumas configurações e escolher a melhor opção com validação cruzada.

O conjunto de teste fica reservado para a avaliação final.

# 10. Avaliação final no conjunto de teste

Depois de treinar e ajustar os modelos, faço a avaliação final no conjunto de teste. Essa etapa mostra como o modelo se comporta em dados que não foram usados no treinamento.

In [ ]:
melhor_nome = comparacao_final.iloc[0]["modelo"]
if melhor_nome == "Regressão Logística Otimizada":
    melhor_modelo = melhor_lr
else:
    melhor_modelo = modelos_treinados[melhor_nome]

print("Melhor modelo por F1 no teste:", melhor_nome)

tabela_final_metricas = resultados_modelos.sort_values(["conjunto", "f1"], ascending=[True, False]).reset_index(drop=True)
display(tabela_final_metricas)

y_pred_test = melhor_modelo.predict(X_test)
y_pred_train = melhor_modelo.predict(X_train)

print("Relatório de classificação - teste")
print(classification_report(y_test, y_pred_test, zero_division=0))

matriz = confusion_matrix(y_test, y_pred_test, labels=[0, 1])
matriz_df = pd.DataFrame(matriz, index=["Real 0", "Real 1"], columns=["Predito 0", "Predito 1"])
display(matriz_df)

sns.heatmap(matriz_df, annot=True, fmt="d", cmap="Blues")
plt.title("Matriz de confusão - melhor modelo no teste")
plt.xlabel("Classe prevista")
plt.ylabel("Classe real")
plt.show()

Na análise de erros, falsos positivos são casos previstos como `1`, mas que eram `0`. Falsos negativos são casos previstos como `0`, mas que eram `1`.

## 10.1 Discussão sobre overfitting e underfitting

Para discutir overfitting e underfitting, considerei a comparação entre as métricas de treino e teste apresentadas na avaliação final.

Não parece ser um caso de **underfitting forte**, porque os modelos superaram o baseline e conseguiram identificar parte importante da classe positiva. Também não há, pela análise geral das métricas, uma conclusão simples de overfitting extremo. Mesmo assim, a comparação precisa ser vista com cuidado, porque a divisão foi temporal: o treino usa processos mais antigos e o teste usa processos mais recentes.

Isso significa que uma diferença entre treino e teste pode acontecer não apenas porque o modelo decorou o treino, mas também porque os processos mais recentes podem ter características diferentes. Essa é uma limitação importante do trabalho e deve ser considerada na interpretação dos resultados.

## 10.2 Análise de erros e limitações

Aqui eu observo os erros do modelo de forma agregada. Não mostro número de processo nem qualquer informação que identifique pessoas.

In [ ]:
erros = X_test.copy()
erros["real"] = y_test.values
erros["predito"] = y_pred_test
erros["tipo_erro"] = np.select(
    [
        (erros["real"].eq(0) & erros["predito"].eq(1)),
        (erros["real"].eq(1) & erros["predito"].eq(0)),
        (erros["real"].eq(erros["predito"])),
    ],
    ["falso_positivo", "falso_negativo", "acerto"],
    default="outro",
)

resumo_erros = erros["tipo_erro"].value_counts().rename_axis("tipo").reset_index(name="processos")
display(resumo_erros)

perfil_erros = (
    erros.query("tipo_erro != 'acerto'")
    .groupby("tipo_erro")
    .agg(
        processos=("real", "size"),
        ano_medio=("ano_ajuizamento", "mean"),
        qtd_assuntos_media=("quantidade_assuntos", "mean"),
        taxa_homicidio_qualificado=("assunto_homicidio_qualificado", "mean"),
        taxa_violencia_domestica=("assunto_violencia_domestica_mulher", "mean"),
        tribunais_distintos=("tribunal", "nunique"),
        classes_distintas=("classe_codigo", "nunique"),
    )
    .reset_index()
)
display(perfil_erros)

for tipo in ["falso_positivo", "falso_negativo"]:
    print(f"\nTop tribunais em {tipo}:")
    display(erros.query("tipo_erro == @tipo")["tribunal"].value_counts().head(8).to_frame("processos"))

# 11. Comparação final dos modelos

A comparação final mostra as principais métricas dos modelos no conjunto de teste. A escolha do melhor modelo considera principalmente o F1-score, mas as outras métricas também ajudam na interpretação.

In [ ]:
# Importância/contribuição das variáveis quando aplicável
try:
    pre = melhor_modelo.named_steps["pre"]
    nomes_features = pre.get_feature_names_out()
    estimador = melhor_modelo.named_steps["model"]
    if hasattr(estimador, "feature_importances_"):
        importancias = pd.DataFrame({"variavel_transformada": nomes_features, "importancia": estimador.feature_importances_})
        display(importancias.sort_values("importancia", ascending=False).head(20))
    elif hasattr(estimador, "coef_"):
        coef = estimador.coef_[0]
        importancias = pd.DataFrame({"variavel_transformada": nomes_features, "coeficiente": coef, "coef_abs": np.abs(coef)})
        display(importancias.sort_values("coef_abs", ascending=False).head(20))
    else:
        print("O estimador selecionado não expõe importância direta de variáveis.")
except Exception as exc:
    print("Não foi possível extrair contribuições das variáveis:", exc)

# 13. Conclusão

A conclusão abaixo usa os resultados calculados pelo próprio notebook. Ela resume o objetivo, o melhor modelo, a comparação com o baseline, as limitações e possíveis próximos passos.

# 12. Boas práticas e rastreabilidade

Alguns cuidados adotados no projeto:

- uso de seed fixa;
- base anonimizada para publicação;
- script para reproduzir a preparação da base;
- uso de pipeline para evitar vazamento;
- teste separado do treino;
- resultados calculados pelo próprio notebook.

# 14. Salvamento de artefatos

Não salvei um modelo final em arquivo porque o treinamento é rápido e o notebook consegue rodar do início ao fim. A base final de modelagem está salva no repositório como `data/mvp/base_modelagem.csv.gz`.

In [ ]:
melhor_teste = comparacao_final.iloc[0]
metricas_melhor_treino = resultados_modelos.query("modelo == @melhor_nome and conjunto == 'treino'").iloc[0]
baseline_teste = resultados_modelos.query("modelo == 'DummyClassifier' and conjunto == 'teste'").iloc[0]
classe_0 = int((y == 0).sum())
classe_1 = int((y == 1).sum())
fp = int(((y_test == 0) & (y_pred_test == 1)).sum())
fn = int(((y_test == 1) & (y_pred_test == 0)).sum())
melhor_configuracao = busca.best_params_

texto_conclusao = f"""
### Conclusão

Neste MVP, eu trabalhei com um problema de classificação binária: prever se um processo relacionado a feminicídio, em primeiro grau, terá decisão judicial substantiva em até 730 dias do ajuizamento. A base final de modelagem contém **{len(df):,} registros** após filtro de primeiro grau, deduplicação, remoção de datas inválidas, exclusão de censurados e retirada de identificadores. A distribuição final tem **{classe_0:,} casos da classe 0** e **{classe_1:,} casos da classe 1**.

Na preparação dos dados, tratei valores ausentes, padronizei variáveis numéricas, transformei variáveis categóricas, agrupei categorias raras, fiz a divisão temporal entre treino e teste e removi variáveis que poderiam causar vazamento. Foram avaliados `DummyClassifier` como baseline, Regressão Logística, Random Forest, Extra Trees, HistGradientBoosting e Regressão Logística otimizada por `GridSearchCV`.

O baseline teve **F1 = {baseline_teste['f1']:.3f}** e **balanced accuracy = {baseline_teste['balanced_accuracy']:.3f}** no teste. A melhor configuração encontrada no tuning foi **{melhor_configuracao}**. O melhor modelo pelo F1 no teste temporal foi **{melhor_nome}**, com **F1 = {melhor_teste['f1']:.3f}**, **recall = {melhor_teste['recall']:.3f}**, **precisão = {melhor_teste['precisao']:.3f}**, **balanced accuracy = {melhor_teste['balanced_accuracy']:.3f}** e **ROC-AUC = {melhor_teste['roc_auc']:.3f}**. No treino, o mesmo modelo teve **F1 = {metricas_melhor_treino['f1']:.3f}**, o que ajuda a avaliar a diferença entre ajuste e generalização.

Escolhi o F1 como métrica principal porque a classe positiva é menor, e olhar só para acurácia poderia dar uma impressão errada do desempenho. Na matriz de confusão do teste, foram observados **{fp:,} falsos positivos** e **{fn:,} falsos negativos**, analisados apenas de forma agregada e sem identificadores.

Considero que o objetivo do MVP foi cumprido como exercício acadêmico: a variável-alvo pôde ser construída de forma defensável com datas de ajuizamento, movimentos de decisão substantiva e tratamento de censura. Ainda assim, o resultado deve ser visto com cautela. Há censura temporal, ausência de processos sigilosos, possível viés de disponibilidade dos dados públicos, diferenças de registro entre tribunais, risco de viés institucional e mudança de distribuição nos anos mais recentes. O modelo identifica associações, não causalidade, e não deve ser usado para decisões individuais.

Próximos passos incluem validar a extração com documentação oficial atualizada do DataJud, ampliar a análise de censura com métodos de sobrevivência, avaliar estabilidade por tribunal e por coorte anual, revisar juridicamente os códigos de decisão e testar novas features iniciais sem introduzir vazamento de dados.
"""

display(Markdown(texto_conclusao))

# 15. Apêndice opcional: Deep Learning, Fine-tuning ou métodos avançados

Não aplicável ao problema. Este MVP usa dados tabulares e modelos clássicos de Machine Learning, então Deep Learning não é necessário.

## Checklist do MVP

- Problema definido: sim.
- Tipo de problema indicado: sim, classificação binária.
- Fonte dos dados descrita: sim, Portal CNJ e API Pública do DataJud/CNJ.
- Base carregada por URL pública: sim.
- Tratamento de censura: sim.
- Prevenção de vazamento: sim.
- Análise exploratória realizada: sim.
- Pipeline de pré-processamento: sim.
- Baseline treinado: sim.
- Modelos candidatos treinados: sim.
- Otimização de hiperparâmetros: sim.
- Avaliação com métricas: sim.
- Matriz de confusão: sim.
- Análise de erros: sim.
- Conclusão com resultados reais: sim.

In [ ]:
# Exemplo opcional, não usado neste MVP:
# import joblib
# joblib.dump(final_model, "modelo_final.pkl")

# 15. Apêndice opcional: Deep Learning, Fine-tuning ou métodos avançados

Não aplicável ao problema, porque este MVP usa dados tabulares e modelos clássicos de Machine Learning. Deep Learning não é necessário para responder ao objetivo do trabalho.

## Checklist do MVP

As respostas abaixo consolidam os requisitos aplicáveis da atividade.

- **O problema está definido?** Sim. O problema é prever a ocorrência de decisão judicial substantiva em até 730 dias do ajuizamento.
- **A tarefa de ML está clara?** Sim. É uma classificação binária supervisionada.
- **A variável-alvo foi definida?** Sim. `decisao_substantiva_ate_730d`, construída com os códigos 219, 220 e 221.
- **A censura foi tratada?** Sim. Processos sem decisão e com menos de 730 dias de acompanhamento foram excluídos da modelagem principal.
- **A unidade de análise foi respeitada?** Sim. Uma linha por processo em primeiro grau após deduplicação.
- **Há prevenção de vazamento?** Sim. Datas de decisão, movimentos posteriores, duração, última atualização, identificadores e resultados judiciais não entram como features.
- **A base pública é compatível com Colab?** Sim. A base é carregada por URL pública com `requests` e `raise_for_status()`.
- **Há EDA inicial?** Sim. O notebook apresenta dimensão, tipos, nulos, duplicidades, estatísticas, alvo, tribunal, ano e variáveis principais.
- **Há preparação dos dados?** Sim. O pipeline trata nulos, escala numéricos, codifica categóricas e agrupa categorias raras.
- **A divisão dos dados foi justificada?** Sim. Foi usada divisão temporal por ser mais coerente com uso futuro.
- **Há baseline?** Sim. `DummyClassifier`.
- **Há modelos candidatos?** Sim. Foram comparados modelos lineares, árvores, ensembles e boosting do `scikit-learn`.
- **Há otimização de hiperparâmetros?** Sim. `GridSearchCV` em Regressão Logística, apenas no treino.
- **Há avaliação no treino e teste?** Sim. Acurácia, precisão, recall, F1, balanced accuracy, ROC-AUC, matriz de confusão e relatório de classificação.
- **Há análise de erros?** Sim. Falsos positivos, falsos negativos e perfis agregados sem identificadores.
- **Há conclusão com resultados reais?** Sim. A conclusão é gerada dinamicamente após a execução das métricas.
- **Deep Learning é aplicável?** Não aplicável ao problema, porque a base é tabular, pequena para modelos clássicos e o objetivo do MVP prioriza interpretabilidade e simplicidade.
- **Uso individual em decisões judiciais é aplicável?** Não aplicável ao problema, porque o modelo é apenas analítico e não deve orientar decisões individuais sobre processos ou pessoas.